In [ ]:
# 1. Remove pacotes antigos do Colab para evitar que o interpretador puxe referências erradas
!pip uninstall -y -q peft torchao

# 2. Instala o Diffusers direto do fonte (exigido pelo script)
!pip install -q git+https://github.com/huggingface/diffusers.git

# 3. Instala o PEFT e atualiza o torchao para a versão estável compatível (>0.16.0)
!pip install -q --upgrade transformers accelerate bitsandbytes peft torchao

# 4. Instala dependências utilitárias restantes
!pip install -q trl torchvision pandas sentencepiece

In [ ]:
%%bash
# Sincroniza o dataset e estrutura do GitHub sem gerar duplicidade de pastas
if [ -d "ceub-modelos-multimodais-ia-generativa" ]; then
  echo "Atualizando repositório existente via Git Pull."
  cd ceub-modelos-multimodais-ia-generativa && git pull
else
  echo "Clonando repositório do zero..."
  git clone https://github.com/alexandregrios/ceub-modelos-multimodais-ia-generativa.git
fi

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

try:
    # Captura a chave dos Secrets do Colab e realiza o login global na sessão
    hf_token = userdata.get('TokenHuggingFace') or userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=True)
    
    # Injeta na variável de ambiente para que ferramentas de CLI a encontrem nativamente
    os.environ["HF_TOKEN"] = hf_token
    print("Autenticação realizada com sucesso! Sessão autorizada para modelos protegidos.")
except Exception:
    print("Erro: Verifique se o segredo 'TokenHuggingFace' está ativo no painel esquerdo do Colab.")

In [ ]:
###Experimento: Tentativa de Fine-Tuning com SDXL 1.0 e Otimização de Hardware

**Descrição do Experimento:**
Esta célula foi desenvolvida como uma tentativa final e estratégica de realizar o fine-tuning (Dreambooth LoRA) do modelo **FLUX.1-dev** (12 bilhões de parâmetros) utilizando recursos limitados de hardware (Google Colab gratuito com GPU NVIDIA T4). 

Para tentar viabilizar a execução sob severa restrição de recursos, foi implementado um arsenal de técnicas de otimização no nível de MLOps:
1. **Otimização de Gradientes ..:** Ativação de `--gradient_checkpointing` e uso do otimizador `--use_8bit_adam` para reduzir a pegada de memória na GPU.
2. **Minimização de Tensores ...:** Redução agressiva da resolução das imagens de treino para `--resolution=256` e configuração de `--gradient_accumulation_steps=1`, visando diminuir drasticamente os buffers de latentes na memória.
3. **Gerenciamento de Cache ....:** Injeção manual de rotinas de coleta de lixo do Python (`gc.collect()`) e esvaziamento do cache da API CUDA do PyTorch (`torch.cuda.empty_cache()`) antes do disparo do processo.

**Diagnóstico Técnico da Falha: OOM de Sistema (CPU) e `SIGKILL: 9`**
Apesar de todas as mitigações aplicadas via software para proteger a VRAM da GPU, o processo foi interrompido abruptamente pelo sistema operacional da máquina virtual, retornando o código de erro **`Signals.SIGKILL: 9`** durante a etapa inicial de carregamento dos fragmentos do modelo (`Loading checkpoint shards: 0% 0/3`).

* **Análise do Gargalo de Infraestrutura:** O script oficial do ecossistema Hugging Face (`diffusers`) tenta instanciar as classes do Transformer e ler as matrizes de pesos brutas na memória RAM do sistema (CPU) antes de transferi-las para a GPU. O arquivo principal do FLUX.1-dev possui cerca de **23.8 GB** descompactados, enquanto o ambiente gratuito do Google Colab oferece um teto rígido de **12.67 GB** de RAM de sistema. 
* **Conclusão:** Ao tentar ler os primeiros shards, o processo ocupou **95.5% da RAM disponível (12.11 GB)**, forçando um *page swapping* severo em disco e o subsequente corte do processo pelo gerenciador do Google (Out of Memory). Diante da inviabilidade física do hardware, a arquitetura do projeto foi refinada de forma madura, pivotando o pipeline para o modelo **Stable Diffusion XL (SDXL)**.

import os
from google.colab import userdata

try:
    meu_token = userdata.get('TokenHuggingFace') or userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = meu_token
except Exception:
    print("Verifique o seu token nos Secrets.")

# Garante o download do script oficial atualizado
!wget -q -O train_dreambooth_lora_flux.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_flux.py

# Limpeza pré-execução
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Executa com todos os limites de otimização ativados
!export HF_TOKEN=$HF_TOKEN && accelerate launch \
  --mixed_precision="fp16" \
  --num_processes=1 \
  --num_machines=1 \
  train_dreambooth_lora_flux.py \
  --pretrained_model_name_or_path="black-forest-labs/FLUX.1-dev" \
  --instance_data_dir="/content/ceub-modelos-multimodais-ia-generativa/dados/imagens" \
  --output_dir="/content/resultados_lora" \
  --instance_prompt="estilo_azulejaria" \
  --resolution=256 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=200 \
  --checkpointing_steps=50 \
  --seed=42 \
  --gradient_checkpointing \
  --use_8bit_adam


In [ ]:
# 1. Remove os scripts antigos do FLUX que estão duplicados na raiz
!rm -f train_dreambooth_lora_flux.py train_dreambooth_lora_flux.py.1

# 2. Limpa a pasta de resultados parciais para garantir que o espaço em disco seja liberado
!rm -rf /content/resultados_lora/*

# 3. Força a limpeza de cache interna do ecossistema Hugging Face (downloads antigos/incompletos)
!rm -rf ~/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev

print("Arquivos desnecessários removidos e espaço em disco liberado!")

In [ ]:
import logging
import warnings

# 1. Configura o silenciamento de logs
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("diffusers").setLevel(logging.ERROR)

# 2. Faz o download do script do Dreambooth LoRA para SDXL
!wget -q -O train_dreambooth_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py

# 3. Dispara o treinamento com a trava de segurança para o disco do Colab
!accelerate launch \
  --num_processes=1 \
  --num_machines=1 \
  --mixed_precision="fp16" \
  --dynamo_backend="no" \
  train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --instance_data_dir="/content/ceub-modelos-multimodais-ia-generativa/dados/imagens" \
  --output_dir="/content/resultados_lora" \
  --mixed_precision="fp16" \
  --instance_prompt="estilo_azulejaria" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --use_8bit_adam \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=800 \
  --checkpointing_steps=200 \
  --checkpoints_total_limit=2 \
  --seed=42 \
  --gradient_checkpointing